In [1]:
#useful Python libraries
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
#sklearn modules
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from xgboost import XGBClassifier

In [2]:
#load my data
df = pd.read_parquet("my_feature_space.parquet")
df

,objectId,finkclass,mean,weighted_mean,standard_deviation,median,amplitude,beyond_1_std,cusum,inter_percentile_range_10,...,magnitude_percentage_ratio_20_10,maximum_slope,median_absolute_deviation,median_buffer_range_percentage_10,percent_amplitude,mean_variance,anderson_darling_normal,chi2,skew,stetson_K
0,ZTF17aaaadkj,CataclyV*,17.222114,17.174713,0.237890,17.224249,1.171396,0.195322,0.108385,0.481224,...,0.676086,410.613047,0.126921,0.457310,1.825376,0.013813,13.875019,242.808733,-2.438629,0.585522
1,ZTF17aaaagyq,CataclyV*,16.735330,16.498882,0.637000,16.862818,1.910264,0.197590,0.152603,1.436352,...,0.429171,354.654346,0.234251,0.414458,2.552179,0.038063,43.600593,2694.289711,-1.706663,0.709195
2,ZTF17aaaaqna,Unknown,14.323769,14.321861,0.206714,14.224896,0.440336,0.169289,0.109419,0.441297,...,0.723493,95.102274,0.062597,0.344004,0.750938,0.014432,138.605800,238.412293,1.464617,0.796598
3,ZTF17aaaarmr,CataclyV*,16.282178,16.256730,0.240892,16.229107,1.577422,0.129736,0.238448,0.418315,...,0.507220,137.920000,0.074253,0.771527,2.849228,0.014795,61.877870,121.862796,5.046224,0.701406
4,ZTF17aaaazob,CataclyV*,17.864093,17.649710,0.403722,17.859806,2.242684,0.179581,0.082159,0.778325,...,0.627876,696.187235,0.198764,0.561076,3.199834,0.022600,49.381505,620.820245,-2.514259,0.488813
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2054,ZTF26aaajqnu,CataclyV*,16.513851,16.289990,0.722135,16.555887,3.018303,0.117801,0.068421,0.868633,...,0.642155,243.547828,0.209332,0.628272,3.193780,0.043729,30.237867,2277.639225,-1.465526,0.593412
2055,ZTF26aaaombw,Unknown,15.516807,15.516408,0.037666,15.522238,0.106918,0.317241,0.289659,0.100583,...,0.651676,83.947826,0.024834,0.230345,0.113693,0.002427,7.927470,12.038277,-0.567986,0.803067
2056,ZTF26aabsgfq,Unknown,14.029994,14.029941,0.029484,14.032178,0.103438,0.282857,0.198469,0.072323,...,0.627597,29.321250,0.017713,0.320000,0.111945,0.002101,1.771802,7.516016,-0.083897,0.769782
2057,ZTF26aaewgqp,Unknown,14.614267,14.612863,0.066856,14.611416,0.766450,0.051392,0.122444,0.084457,...,0.658268,181.159533,0.022746,0.966809,1.428257,0.004575,76.158691,14.791354,12.549345,0.479608


In [3]:
# 1 = CataclyV (including candidates)
# 0 = everything else

#y_true = df['finkclass'].apply(
 #   lambda x: 1 if 'Cat' in str(x) else 0     #true class title
#).values

In [4]:
# Load saved model
#bestmodel at first
best_model = joblib.load("cv_classifier_best_model_12_1.pkl")

In [5]:
#feature space
feature_columns = df.drop(columns=['objectId', 'finkclass']).columns #drop non_numeric columns

X = df[feature_columns].values #feature column

In [6]:
y_pred = best_model.predict(X) #predict model
print("Unique predictions:", np.unique(y_pred)) #cross check

Unique predictions: [0 1]


In [7]:
# Check class probabilities for the test set.
# predict_proba returns an array of shape (N_test, 2),
# where N_test is the number of test objects.
# Column 0 : probability of class 0 (negative, unknown)
# Column 1 : probability of class +1 (positive, cvs)
probs = best_model.predict_proba(X)
print('probs',probs)

probs [[0.4283265  0.5716735 ]
 [0.8602915  0.13970852]
 [0.9360449  0.06395514]
 ...
 [0.6903353  0.30966476]
 [0.9968498  0.00315024]
 [0.9541829  0.04581705]]


In [8]:
# CV probability
cv_prob = probs[:,1]

# create dataframe for attaching objectid
prob_df = pd.DataFrame({
    "objectId": df["objectId"],
    "cv_probability": cv_prob
})
#check output
print(prob_df.head())

       objectId  cv_probability
0  ZTF17aaaadkj        0.571674
1  ZTF17aaaagyq        0.139709
2  ZTF17aaaaqna        0.063955
3  ZTF17aaaarmr        0.045247
4  ZTF17aaaazob        0.344255


In [9]:
# CV probability with finkclass_column_atached
cv_prob = probs[:,1]

# create dataframe with required columns
prob_df = pd.DataFrame({
    "objectId": df["objectId"],
    "finkclass": df["finkclass"],
    "cv_probability": cv_prob
})

# check
print(prob_df.head())

       objectId  finkclass  cv_probability
0  ZTF17aaaadkj  CataclyV*        0.571674
1  ZTF17aaaagyq  CataclyV*        0.139709
2  ZTF17aaaaqna    Unknown        0.063955
3  ZTF17aaaarmr  CataclyV*        0.045247
4  ZTF17aaaazob  CataclyV*        0.344255


In [10]:
#saving into file
prob_df.to_csv("best_model_prediction_RS_12_1.csv", index=False)

In [11]:
prob_df

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.571674
1,ZTF17aaaagyq,CataclyV*,0.139709
2,ZTF17aaaaqna,Unknown,0.063955
3,ZTF17aaaarmr,CataclyV*,0.045247
4,ZTF17aaaazob,CataclyV*,0.344255
...,...,...,...
2054,ZTF26aaajqnu,CataclyV*,0.792732
2055,ZTF26aaaombw,Unknown,0.066021
2056,ZTF26aabsgfq,Unknown,0.309665
2057,ZTF26aaewgqp,Unknown,0.003150


In [12]:
#probablity distribution checking
high_prob = prob_df[prob_df["cv_probability"] > 0.9]
print("Number of objects with CV probability > 0.9:", len(high_prob))

Number of objects with CV probability > 0.9: 281


In [13]:
#saving into parquet
high_prob.to_csv("best_model_prediction_cv_candidates_above_0.9_RS_12_1.csv", index=False)

In [14]:
high_prob

,objectId,finkclass,cv_probability
6,ZTF17aaabavb,CataclyV*,0.979772
7,ZTF17aaabfay,CataclyV*,0.945092
17,ZTF17aaaehby,CataclyV*,0.926704
23,ZTF17aaaeslm,CataclyV*,0.991315
37,ZTF17aaagywy,CataclyV*,0.981443
...,...,...,...
2034,ZTF25aagjobe,Unknown,0.929035
2035,ZTF25aaguzft,CataclyV*,0.988532
2036,ZTF25aagvjel,CataclyV*,0.977606
2045,ZTF25abungcd,CataclyV*,0.989311


In [15]:
#now for previously saved model
# Load saved model
model = joblib.load("cv_classifier_xgb_boost_biggie_set_12_1.pkl")

In [16]:
y_pred_1 = model.predict(X) #predict model

In [17]:
y_pred_1

array([1, 0, 0, ..., 0, 0, 0], shape=(2059,))

In [18]:
# Check class probabilities for the test set.
# predict_proba returns an array of shape (N_test, 2),
# where N_test is the number of test objects.
# Column 0 : probability of class 0 (negative, unknown)
# Column 1 : probability of class +1 (positive, cvs)
probs_1 = model.predict_proba(X)
print('probs_1',probs_1)

probs_1 [[0.05658126 0.94341874]
 [0.5931992  0.4068008 ]
 [0.97403127 0.0259687 ]
 ...
 [0.9630023  0.03699771]
 [0.9290233  0.07097667]
 [0.9038817  0.09611826]]


In [19]:
# CV probability
cv_prob_1 = probs_1[:,1]

# create dataframe for attaching objectid
prob_df_1 = pd.DataFrame({
    "objectId": df["objectId"],
    "finkclass": df["finkclass"],
    "cv_probability": cv_prob_1
})
#check output
print(prob_df_1.head())

       objectId  finkclass  cv_probability
0  ZTF17aaaadkj  CataclyV*        0.943419
1  ZTF17aaaagyq  CataclyV*        0.406801
2  ZTF17aaaaqna    Unknown        0.025969
3  ZTF17aaaarmr  CataclyV*        0.318796
4  ZTF17aaaazob  CataclyV*        0.838113


In [20]:
#saving into parquet
prob_df_1.to_csv("class prediction for previousy saved model_RS_12_1.csv", index=False)

In [21]:
prob_df_1

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.943419
1,ZTF17aaaagyq,CataclyV*,0.406801
2,ZTF17aaaaqna,Unknown,0.025969
3,ZTF17aaaarmr,CataclyV*,0.318796
4,ZTF17aaaazob,CataclyV*,0.838113
...,...,...,...
2054,ZTF26aaajqnu,CataclyV*,0.905378
2055,ZTF26aaaombw,Unknown,0.031484
2056,ZTF26aabsgfq,Unknown,0.036998
2057,ZTF26aaewgqp,Unknown,0.070977


In [22]:
#probablity distribution checking
high_prob_1 = prob_df_1[prob_df_1["cv_probability"] > 0.9]
print("Number of objects with CV probability > 0.9:", len(high_prob_1))

Number of objects with CV probability > 0.9: 174


In [23]:
#saving into parquet
high_prob_1.to_csv("cv_candidates_above_0.9_for_previously_saved_model_RS_12_1.csv", index=False)

In [24]:
high_prob_1

,objectId,finkclass,cv_probability
0,ZTF17aaaadkj,CataclyV*,0.943419
13,ZTF17aaadasj,CataclyV*,0.969983
36,ZTF17aaagyuc,CataclyV*,0.902827
37,ZTF17aaagywy,CataclyV*,0.938527
52,ZTF17aaaiwfl,CataclyV*,0.907051
...,...,...,...
2022,ZTF24abxhwsr,Unknown,0.933623
2023,ZTF24abyucsd,CataclyV*,0.980163
2027,ZTF25aaapxey,CataclyV*,0.965769
2045,ZTF25abungcd,CataclyV*,0.970175


In [25]:
#compare b/w them
y_pred_1 = best_model.predict(X)
y_pred_2 = model.predict(X)

In [26]:
##accuracy score
#print("best_model:", accuracy_score(y_true, y_pred_1))
#print("model:", accuracy_score(y_true, y_pred_2))

In [27]:
#find the mismatch
#load high probability files
high_prob_best = pd.read_csv(
    "best_model_prediction_cv_candidates_above_0.9_RS_12_1.csv"
)

high_prob_old = pd.read_csv(
    "cv_candidates_above_0.9_for_previously_saved_model_RS_12_1.csv"
)

In [28]:
#mismatches
cv_best = set(high_prob_best["objectId"])
cv_old = set(high_prob_old["objectId"])

only_in_best = cv_best - cv_old
only_in_old = cv_old - cv_best

print("Only in best model:", len(only_in_best))
print("Only in old model:", len(only_in_old))

Only in best model: 183
Only in old model: 76


In [29]:
#extra objects of new model
extra_best = high_prob_best[
    high_prob_best["objectId"].isin(only_in_best)
]
#extra objects of old model
extra_old = high_prob_old[
    high_prob_old["objectId"].isin(only_in_old)
]
#print
print(extra_best)
print(extra_old)

         objectId  finkclass  cv_probability
0    ZTF17aaabavb  CataclyV*        0.979772
1    ZTF17aaabfay  CataclyV*        0.945092
2    ZTF17aaaehby  CataclyV*        0.926704
3    ZTF17aaaeslm  CataclyV*        0.991315
5    ZTF17aaahrrt  CataclyV*        0.925403
..            ...        ...             ...
272  ZTF24aayrves    Unknown        0.964866
276  ZTF25aagjobe    Unknown        0.929036
277  ZTF25aaguzft  CataclyV*        0.988532
278  ZTF25aagvjel  CataclyV*        0.977606
280  ZTF26aaaelef  CataclyV*        0.909267

[183 rows x 3 columns]
         objectId  finkclass  cv_probability
0    ZTF17aaaadkj  CataclyV*        0.943419
1    ZTF17aaadasj  CataclyV*        0.969983
2    ZTF17aaagyuc  CataclyV*        0.902827
4    ZTF17aaaiwfl  CataclyV*        0.907051
9    ZTF17aaajnpg  CataclyV*        0.920934
..            ...        ...             ...
158  ZTF20aaklbiy  CataclyV*        0.925449
161  ZTF20adcaqij  CataclyV*        0.922216
162  ZTF21aaeztlf  CataclyV*   

In [30]:
#coomon elements
common_ids = cv_best.intersection(cv_old)

print("Common CV candidates:", len(common_ids))

Common CV candidates: 98


In [31]:
#change in probability
common_best = high_prob_best[
    high_prob_best["objectId"].isin(common_ids)
]

common_old = high_prob_old[
    high_prob_old["objectId"].isin(common_ids)
]

merged = common_best.merge(
    common_old,
    on="objectId",
    suffixes=("_best", "_old")
)

merged["prob_diff"] = abs(
    merged["cv_probability_best"]
    - merged["cv_probability_old"]
)

In [32]:
#show
merged.sort_values(
    "prob_diff",
    ascending=True
)

,objectId,finkclass_best,cv_probability_best,finkclass_old,cv_probability_old,prob_diff
9,ZTF17aabtlxj,CataclyV*,0.933654,CataclyV*,0.934067,0.000412
46,ZTF18acpdnmd,Unknown,0.942329,Unknown,0.941328,0.001000
31,ZTF18abtlkco,CataclyV*,0.979359,CataclyV*,0.978272,0.001087
37,ZTF18abwpsyj,CataclyV*,0.910962,CataclyV*,0.912055,0.001093
34,ZTF18abvfyyr,CataclyV*,0.943824,CataclyV*,0.942713,0.001111
...,...,...,...,...,...,...
17,ZTF18aaawnsj,CataclyV*,0.973148,CataclyV*,0.902128,0.071020
41,ZTF18accdkuh,Unknown,0.986102,Unknown,0.911297,0.074805
76,ZTF19aakuyyr,CataclyV*,0.977339,CataclyV*,0.900778,0.076561
91,ZTF22aagnmun,Unknown,0.989945,Unknown,0.902872,0.087073
